Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise: Data Modeling with Apache Iceberg Tables

The fastest query is the one that never reads from disk. Iceberg data modeling is about laying out files so the engine can **skip** them using partition values and per-file stats (min/max, null/NaN counts, record counts) stored in metadata — without opening them.

In this exercise:
- Work with the NYC Taxi dataset
- Build tables with 4 partitioning strategies + sort orders
- Inspect metadata to see how layout changes
- Test predicate pushdown across query patterns
- Read the Spark UI to see file skipping in action

⚠️ This environment uses **Spark Connect**: one Spark server runs in the background, notebooks connect as thin clients. If `ConnectionRefusedError` appears, check `docker logs jupyter-spark` or restart with `docker compose restart jupyter`.

## Initialize Spark Session

First run may be slow while dependencies download.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("DataModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

Spark 4.0.1 initialized!


## Create Namespace


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")
print("Namespace 'taxi' created!")

Namespace 'taxi' created!


In [4]:
spark.sql("SHOW NAMESPACES IN polaris").show()


+---------+
|namespace|
+---------+
|     demo|
|     taxi|
+---------+



## Download NYC Taxi Data

Yellow Taxi trips, **June–October 2023** (~225 MB, 5 months). Gives enough volume to see real differences between partitioning strategies.

In [5]:
import urllib.request
import os
import boto3
from botocore.client import Config

# Configure MinIO client using S3-compatible API
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id=os.environ.get('MINIO_ROOT_USER', 'admin'),
    aws_secret_access_key=os.environ.get('MINIO_ROOT_PASSWORD', 'password'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download NYC Yellow Taxi data for June through October 2023 and upload to MinIO
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
minio_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

# Create temp directory
os.makedirs(temp_dir, exist_ok=True)

# Download data for months 6-10 (June through October)
months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    minio_key = f"{minio_prefix}/{filename}"
    
    try:
        # Check if file already exists locally
        if os.path.exists(local_path):
            print(f"{filename} already exists locally, skipping download")
        else:
            # Download file
            print(f"Downloading {filename} (~45MB)...")
            urllib.request.urlretrieve(url, local_path)
            print(f"  Downloaded to {local_path}")
        
        # Upload to MinIO
        print(f"  Uploading to MinIO: s3a://{bucket_name}/{minio_key}...")
        s3_client.upload_file(local_path, bucket_name, minio_key)
        print(f"  Uploaded successfully!")
        uploaded_files.append(minio_key)
        
        # Clean up local file
        os.remove(local_path)
    except Exception as e:
        print(f"  Error processing {filename}: {e}")

print(f"\nAll files ready in MinIO! Total: {len(uploaded_files)} months of data")
print(f"Location: s3a://{bucket_name}/{minio_prefix}/")

  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-06.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-06.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-07.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-07.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-08.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-08.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-09.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-09.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-10.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-10.parquet...
  Uploaded successfully!

All files ready in MinIO! Total: 5 months of data
Location: s3a://warehouse/raw/


## Load and Explore the Data

Load the Parquet files and inspect schema + sample rows.

In [6]:
# Read all parquet files from MinIO
s3_path = "s3a://warehouse/raw/"

taxi_df = spark.read.parquet(s3_path)

print(f"Reading from: {s3_path}")
print(f"Total records: {taxi_df.count():,}")
print(f"\nSchema:")
taxi_df.printSchema()

Reading from: s3a://warehouse/raw/
Total records: 15,407,558

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [7]:
# Show sample data
print("Sample records:")
taxi_df.show(5, truncate=False)

Sample records:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2023-10-01 00:16:44 |2023-10-01 00:16:49  |1              |0.0          |1         |N                 |168         |168         |2           |3.0        |1.0  |0

In [8]:
# Get date range and basic statistics
print("Date range:")
taxi_df.select(
    F.min("tpep_pickup_datetime").alias("min_date"),
    F.max("tpep_pickup_datetime").alias("max_date")
).show()

print("\nBasic statistics:")
taxi_df.select(
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("total_amount"), 2).alias("avg_fare"),
    F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("distinct_days")
).show()

Date range:
+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2002-12-31 22:27:05|2023-11-23 13:40:43|
+-------------------+-------------------+


Basic statistics:
+-----------+------------+--------+-------------+
|total_trips|avg_distance|avg_fare|distinct_days|
+-----------+------------+--------+-------------+
|   15407558|        4.35|   29.05|          161|
+-----------+------------+--------+-------------+



## Creating Iceberg Tables with Different Partitioning Specs

Iceberg partitions via **transform functions** (e.g. `months()`, `days()`) applied to existing columns — no synthetic partition columns, no manual values. This is **hidden partitioning**: queries filter on the raw column and Iceberg applies the transform internally to match the partition scheme.

We'll use one spec per table (via `CREATE TABLE AS SELECT`) to compare effects. Target file size is pinned to **16 MB** here to make partitioning effects visible.

→ Deeper background: [`docs/03-partitioning.md`](../docs/03-partitioning.md)

## Strategy 1: Unpartitioned Table

Baseline — no partitioning.

In [9]:
# Create unpartitioned table using CTAS
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Unpartitioned table created!")

Unpartitioned table created!


### Analyze Unpartitioned Table Metadata

Iceberg exposes [**metadata tables**](https://iceberg.apache.org/docs/latest/spark-queries/#inspecting-tables) to inspect physical layout without scanning data. In Spark: `catalog.namespace.table.metadata_table`.

In [10]:
# Check the files metadata table
# readable_metrics is an Iceberg metadata column containing per-file statistics
# (lower_bound, upper_bound, null_count, nan_count, etc.) for every data column.
# These stats let query engines skip files without opening them.
print("Files in unpartitioned table:")
spark.sql("""
    SELECT 
        file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count,
        readable_metrics.total_amount.lower_bound as total_amount_min,
        readable_metrics.total_amount.upper_bound as total_amount_max
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

Files in unpartitioned table:
+----------------------------------------------------------------------------------------------------------+------------------+------------+----------------+----------------+
|file_path                                                                                                 |size_mb           |record_count|total_amount_min|total_amount_max|
+----------------------------------------------------------------------------------------------------------+------------------+------------+----------------+----------------+
|s3://warehouse/taxi/trips_unpartitioned/data/00000-23-8bfd1b03-2e9c-4a4e-bef9-dce21f805c1b-0-00001.parquet|16.07091236114502 |1058000     |-661.0          |673.75          |
|s3://warehouse/taxi/trips_unpartitioned/data/00000-23-8bfd1b03-2e9c-4a4e-bef9-dce21f805c1b-0-00002.parquet|15.988000869750977|1059000     |-819.75         |819.75          |
|s3://warehouse/taxi/trips_unpartitioned/data/00000-23-8bfd1b03-2e9c-4a4e-bef9-dce21f805c1b-0-0

In [11]:
# The entries metadata table exposes manifest-level entries (one row per data file
# per manifest). It contains the same information as the files table but parallelizes
# better in Spark because each manifest can be read independently.
spark.sql("""
    SELECT 
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_unpartitioned.entries
""").show()

+------------+----------+
|record_count|data_files|
+------------+----------+
|    15407558|        17|
+------------+----------+



In [12]:
# Check snapshots
print("Snapshots:")
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_unpartitioned.snapshots
""").show(truncate=False)

Snapshots:
+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-08-10 10:16:02.562|1242379465758984597|append   |
+-----------------------+-------------------+---------+



### Check MinIO for Unpartitioned Table

**MinIO:** http://localhost:9001/browser/warehouse/taxi/trips_unpartitioned/ · `admin / password`

`data/` holds the Parquet files referenced above; `metadata/` holds `metadata.json`, manifests (`*m*.avro`), and manifest lists (`snap*.avro`).

## Strategy 2: Monthly Partitioning

Partition by **month** via `months()`. All rows sharing a month value land in the same files, so time-based queries only touch the months they need.

In [5]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Monthly partitioned table created!")

Monthly partitioned table created!


### Analyze Monthly Partitioned Table

In [6]:
print("Files in monthly partitioned table:")
spark.sql("""
    SELECT 
        file_path,
        partition,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count
    FROM polaris.taxi.trips_by_month.files
""").show(truncate=False)

Files in monthly partitioned table:
+----------------------------------------------------------------------------------------------------------------------------------------+---------+--------------------+------------+
|file_path                                                                                                                               |partition|size_mb             |record_count|
+----------------------------------------------------------------------------------------------------------------------------------------+---------+--------------------+------------+
|s3://warehouse/taxi/trips_by_month/data/tpep_pickup_datetime_month=2023-11/00000-38-25e5c4d9-2144-4f35-a2f6-4e7f00823644-0-00001.parquet|{646}    |0.006367683410644531|13          |
|s3://warehouse/taxi/trips_by_month/data/tpep_pickup_datetime_month=2023-07/00000-38-25e5c4d9-2144-4f35-a2f6-4e7f00823644-0-00002.parquet|{642}    |16.1483097076416    |1038000     |
|s3://warehouse/taxi/trips_by_month/data/tpep_pic

In [7]:
# Check partition statistics
# Note we are using the "entries" table because it parallelizes better on Spark than the "partitions" table
# The partition values below (e.g., 641, 642) are month offsets from epoch (Jan 1970 = 0).
# So 641 = June 2023, 642 = July 2023, etc. The next cell explains this encoding.
print("Partition statistics:")
spark.sql("""
    SELECT 
        data_file.partition,
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_month.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
""").show()

Partition statistics:
+---------+------------+----------+
|partition|record_count|data_files|
+---------+------------+----------+
|    {395}|           5|         1|
|    {396}|           1|         1|
|    {467}|          10|         1|
|    {468}|          11|         1|
|    {640}|          19|         1|
|    {641}|     3307234|         4|
|    {642}|     2907084|         3|
|    {643}|     2824199|         3|
|    {644}|     2846740|         3|
|    {645}|     3522242|         4|
|    {646}|          13|         1|
+---------+------------+----------+



### Check MinIO for Monthly Partitioned Table

**MinIO:** http://localhost:9001/browser/warehouse/taxi/trips_by_month/

- `months()` returns an integer = months since epoch (Jan 1970 = 0). So `{641}` = June 2023, `{642}` = July 2023… Engines translate this automatically; you never use these numbers in queries.
- The `tpep_pickup_datetime_month=2023-06/` directories are for **human browsing only**. Iceberg tracks partitions in manifest metadata, not directory paths — query planning ignores the directory layout.

## Strategy 3: Daily Partitioning

`days()` → one file per day. Very selective (each file ≈ 1/30 of a monthly file), but most engines struggle with many small files. We'll fix that in later modules.

In [18]:
# Create daily partitioned table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Daily partitioned table created!")

Daily partitioned table created!


### Analyze Daily Partitioned Table

In [ ]:
# Check partition statistics - note we have many more partitions now
print("Partition statistics (showing first 10):")
spark.sql("""
    SELECT 
        data_file.partition,
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_day.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
    LIMIT 10
""").show()

Partition statistics (showing first 10):


In [20]:
# Count total partitions
partition_count = spark.sql("""
    SELECT COUNT(DISTINCT data_file.partition) as partition_count 
    FROM polaris.taxi.trips_by_day.entries
""").collect()[0][0]

print(f"Total partitions in daily partitioned table: {partition_count}")

Total partitions in daily partitioned table: 161


In [21]:
# Check file distribution across partitions
print("File count per partition (sample):")
spark.sql("""
    SELECT 
        data_file.partition,
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(SUM(data_file.file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_by_day.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
    LIMIT 10
""").show()

File count per partition (sample):
+------------+----------+-------------+-------------+
|   partition|file_count|total_records|total_size_mb|
+------------+----------+-------------+-------------+
|{2002-12-31}|         1|            5|         0.01|
|{2003-01-01}|         1|            1|         0.01|
|{2008-12-31}|         1|           10|         0.01|
|{2009-01-01}|         1|           11|         0.01|
|{2023-05-31}|         1|           19|         0.01|
|{2023-06-01}|         1|       123258|         1.85|
|{2023-06-02}|         1|       118971|          1.8|
|{2023-06-03}|         1|       118159|         1.79|
|{2023-06-04}|         1|       100293|         1.59|
|{2023-06-05}|         1|       103470|         1.61|
+------------+----------+-------------+-------------+



### Check MinIO for Daily Partitioned Table

**MinIO:** http://localhost:9001/browser/warehouse/taxi/trips_by_day/

## What shouldn't we partition on?

Avoid high-cardinality columns — they create many tiny files. Query cost scales with **file count** and data read; the per-file overhead (open/read-metadata/close) dominates when files are a few MB. Rough target: **100 MB – 1 GB per partition**. Daily partitioning here produced ~1.8 MB files — too small.

## Strategy 4: Monthly Partition + Sort Order

A **sort order** controls physical layout within files. Sorting on a column tightens its min/max bounds per file, so metric pushdown can skip files whose range doesn't overlap the filter. Cost: slower writes, and Iceberg doesn't re-sort on later inserts — needs maintenance.

Here: partition by month, sort by pickup location.

In [ ]:
# Create monthly partitioned table with sort order on pickup location
# Important: Set sort order BEFORE inserting data so it applies during write
#
# Spark SQL doesn't support setting sort order in a CTAS statement, so we use a
# three-step workaround: create empty table, set sort order, then insert data.
# This is a Spark limitation, not a fundamental Iceberg pattern.

# Step 1: Create empty table structure (using CTAS with false condition)
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_sorted """)
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_sorted
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
    WHERE 1 = 0
""")

# Step 2: Set the sort order (must be done before data is written)
spark.sql("""
    ALTER TABLE polaris.taxi.trips_sorted
    WRITE ORDERED BY PULocationID
""")

print("Table structure created with sort order configured")

# Step 3: Insert data - it will be sorted as it's written
spark.sql("""
    INSERT INTO polaris.taxi.trips_sorted
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Sorted table populated!")

### Analyze Sorted Table

Each file now covers an exclusive range of `PULocationID` — great for location filters. Note the write cost, and that sort efficiency degrades across inserts without maintenance.

In [ ]:
# Check file metadata with bounds
print("File statistics (showing readable metrics including min/max for PULocationID):")
spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        readable_metrics.PULocationID.lower_bound as PULocationID_MIN,
        readable_metrics.PULocationID.upper_bound as PULocationID_MAX
    FROM polaris.taxi.trips_sorted.files
    ORDER BY partition DESC, PULocationID_MIN
    LIMIT 10
""").show(truncate=False)

## Query Testing: Pushdown Optimizations

Iceberg evaluates predicates against metadata to decide which files to scan. Recall the metadata tree: **snapshot → manifest list → manifests → data files**.

1. **Manifest level** — partition predicates compared to manifest summaries → skip whole manifests (critical at scale).
2. **File level** — surviving manifests are opened; each data file's partition value + column stats (min/max) compared to both partition and column predicates.

→ Deep dive: [`docs/04-query-optimization.md`](../docs/04-query-optimization.md)

### Query 1: No Predicate Pushdown (Full Scan)

No filters → every file must be read.

In [ ]:
# Query without any filters - full scan
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
""")

print("Full scan query results:")
result.show()


**Check Spark UI → http://localhost:4040/SQL/ → find query → Description → showString → `BatchScan polaris.taxi.trips_by_day`**

Watch: `number of result data files`, `number of skipped data files` (should be 0), `number of skipped data manifests` (0), `total data file size (bytes)`, `number of output rows`.

### Query 2: Partition Pushdown

Filter by date → Iceberg skips entire partitions.

In [ ]:
# Query with date filter - partition pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""")

print("Partition pushdown query results (August 2023 only):")
result.show()


**Check Spark UI → `BatchScan polaris.taxi.trips_by_day`**

- `number of result data files`: ~31 (one per August day)
- `number of skipped data files`: other months
- `number of skipped data manifests`: other partitions
- `total data file size (bytes)`: far smaller than full scan
- `total planning duration (ms)`

→ **Partition pruning skipped ~4/5 of files from metadata alone.**

### Query 3: Metric Pushdown (Min/Max)

Filter by a non-partition column with a range predicate → Iceberg uses file-level min/max stats to skip files.

In [ ]:
# Query with location filter - metric pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE PULocationID IN (132, 138, 161, 230, 237)
""")

print("Metric pushdown query results (specific locations):")
result.show()


**Check Spark UI → `BatchScan polaris.taxi.trips_sorted`**

- `number of skipped data files`: files whose `PULocationID` min/max don't overlap the filter
- `number of scanned data manifests`, `total data file size (bytes)`

→ **Sort order + metric pushdown skipped files via min/max bounds.**

### Query 4: Combined Partition + Metric Pushdown

Both: partition pruning (month) + metric filtering (location).

In [ ]:
# Query with both month and location filters
result = spark.sql("""
    SELECT 
        COUNT(*) as matching_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
      AND PULocationID IN (132, 138, 161, 230, 237)
""")

print("Combined pushdown query results (August 2023, specific locations):")
result.show()


**Check Spark UI → `BatchScan polaris.taxi.trips_sorted`**

- `number of result data files`: minimal — matches BOTH filters
- `number of skipped data files`: maximum (partition + metrics)
- `number of skipped data manifests`: eliminated by partition filter
- `total data file size (bytes)`: smallest scan

→ **Best case: partition pruning removed months, then metric pushdown removed files inside August.**

## Performance Comparison: All Strategies

A helper runs the same query against all four tables and reports timing. Performance has two costs: **opening files** (per-file overhead) and **reading data**. Same data volume can still differ if one version has many small files. Engine-dependent — run each query a few times to skip warmup.

In [ ]:
import time

def compare_partitioning_strategies(where_clause, description):
    """
    Compare query performance across all partitioning strategies.
    
    Args:
        where_clause: SQL WHERE clause (without the WHERE keyword)
        description: Human-readable description of the query
    """
    # List of all tables to compare
    tables = [
        ('trips_unpartitioned', 'Unpartitioned'),
        ('trips_by_month', 'Monthly Partitions'),
        ('trips_by_day', 'Daily Partitions'),
        ('trips_sorted', 'Monthly + Sorted by Location')
    ]
    
    results = []
    
    print(f"Query: {description}")
    print(f"Predicate: {where_clause}")
    print("\n" + "=" * 80)
    
    for table_name, table_desc in tables:
        # Build the query with the table name substituted
        query = f"""
            SELECT COUNT(*) as trip_count,
                   ROUND(AVG(total_amount), 2) as avg_fare
            FROM polaris.taxi.{table_name}
            WHERE {where_clause}
        """
        
        start = time.time()
        result = spark.sql(query).collect()
        elapsed = time.time() - start
        
        count = result[0][0]
        avg_fare = result[0][1] if result[0][1] else 0.0
        results.append((table_desc, count, avg_fare, elapsed))
        
        print(f"{table_desc:30} | Trips: {count:>6,} | Avg: ${avg_fare:>6.2f} | Time: {elapsed:>6.3f}s")
    
    print("=" * 80)
    
    # Calculate and display speedups
    baseline_time = results[0][3]
    print("\nSpeedups vs Unpartitioned:")
    for table_desc, count, avg_fare, elapsed in results[1:]:
        speedup = baseline_time / elapsed if elapsed > 0 else 0
        print(f"  {table_desc:30} : {speedup:>5.2f}x faster")
    
    print("\n" + "=" * 80 + "\n")
    
    return results

Use `compare_partitioning_strategies(where_clause, description)` to run the same query against all four tables.

### Example 1: Time Range + Location

3 days in mid-August + a specific pickup location. Iceberg converts this predicate to match each table's spec automatically.

In [ ]:
# Example 1: 3 days + specific location
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-08-14' AND tpep_pickup_datetime < '2023-08-17' AND PULocationID = 237",
    description="3 days in mid-August, pickup location 237"
)

**Analysis:**
- **Unpartitioned**: scans all ~153 files
- **Monthly**: August files only
- **Daily**: 3 files
- **Monthly + Sorted**: a single August file

### Example 2: Full Month Range

A whole month — partition pruning without day-level granularity.

In [ ]:
# Example 2: Full month
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-07-01' AND tpep_pickup_datetime < '2023-08-01'",
    description="All of July 2023"
)

**Analysis:**
- Monthly and Monthly+Sorted prune identically here (same file count). Sort on `PULocationID` adds nothing — the query has no predicate on the sort column; timing differences are noise.
- Daily still opens ~31 small files (per-file overhead), so it prunes less efficiently for a full month.
- Both partitioned versions beat unpartitioned.

### Example 3: Single Day + Location Range

Very selective: one day, multiple locations. Shows metric pushdown.

In [ ]:
# Example 3: Single day + location range
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-08-15' AND tpep_pickup_datetime < '2023-08-16' AND PULocationID IN (132, 138, 161)",
    description="Single day (Aug 15), 3 specific locations"
)

**Analysis:**
- Unpartitioned still benefits from column-metric pruning (no explicit partition).
- Monthly must read all month files.
- Daily wins — most selective partition pruning.

### Example 4: Location-Only Filter (No Time)

No time predicate → shows the value of partitioning on the query column.

In [ ]:
# Example 4: Location only (no time filter)
compare_partitioning_strategies(
    where_clause="PULocationID = 237",
    description="All trips from location 237 (no time filter)"
)

**Analysis:**
- Only column-metric pruning available.
- **Sorted table** wins big via metric pushdown on `PULocationID`.
- Daily ends up slower than unpartitioned (file-open cost).

### Try Your Own Query

Experiment with time ranges (hours/days/weeks/months), location filters, and other columns like `total_amount`, `trip_distance`, `passenger_count`.

In [ ]:
# Your custom query here!
# Uncomment and modify:

# compare_partitioning_strategies(
#     where_clause="tpep_pickup_datetime >= '2023-09-01' AND tpep_pickup_datetime < '2023-09-02' AND total_amount > 50",
#     description="High-value trips on Sept 1"
# )

### Key Takeaways

1. **Hidden partitioning** — never specify the transform in queries; Iceberg matches it automatically.
2. **File count cuts both ways** — partitions help selective queries but small files hurt non-selective ones.
3. **Sort orders enable metric pushdown** — file-level filtering even without partitioning on that column.
4. **Pushdowns are query-dependent** — no benefit if the query doesn't use the partition transform or a column metric.

Check **Spark UI → SQL** for each query to see the actual file statistics.

## Summary

**Partitioning:** mind cardinality, match query patterns, transforms give flexibility.
**Sort orders:** help range queries + metric pushdown; use for frequent `WHERE` columns.
**Metadata tables:** `.files` (layout + stats), `.snapshots` (commits).
**Pushdowns:** partition pruning (manifests/files) + metric pushdown (min/max).

**Best practices:** partition by query patterns · avoid over-partitioning · target 100 MB–1 GB/partition · lean on hidden partitioning · sort for range queries · check metadata tables before/after changes.

→ Cheatsheet: [`docs/08-cheatsheet.md`](../docs/08-cheatsheet.md)